In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 📥 Load Dataset
We use CIFAR-10, which contains 60,000 color images of size 32×32×3.

50,000 training images
10,000 test images

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

class_names = [
    'airplane','automobile','bird','cat','deer',
    'dog','frog','horse','ship','truck'
]

print(x_train.shape)
print(x_test.shape)

In [ ]:
plt.figure(figsize=(10,5))
for i in range(10):
    plt.subplot(2,5,i+1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i][0]])
    plt.axis("off")
plt.tight_layout()
plt.show()

# 🧹 Preprocessing
We normalize pixel values from **0–255 → 0–1** so training becomes stable.

In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train_flat = x_train.reshape(len(x_train),3072)
x_test_flat = x_test.reshape(len(x_test),3072)

# 🔹 Part 1: ANN Model


In [ ]:
ann_model = models.Sequential([
    layers.Input(shape=(3072,)),
    layers.Dense(512,activation='relu'),
    layers.Dropout(0.3),

    layers.Dense(256,activation='relu'),
    layers.Dropout(0.3),

    layers.Dense(128,activation='relu'),

    layers.Dense(10,activation='softmax')
])

In [ ]:
ann_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [ ]:
ann_history = ann_model.fit(
    x_train_flat,
    y_train,
    epochs=20,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop]
)

# Evaluate

In [ ]:
ann_loss, ann_acc = ann_model.evaluate(x_test_flat,y_test)

# 🔹 Part 2: CNN Model


In [ ]:
cnn_model=models.Sequential([

layers.Input(shape=(32,32,3)),

layers.Conv2D(32,(3,3),activation='relu',padding='same'),
layers.BatchNormalization(),
layers.MaxPooling2D(),

layers.Conv2D(64,(3,3),activation='relu',padding='same'),
layers.BatchNormalization(),
layers.MaxPooling2D(),

layers.Conv2D(128,(3,3),activation='relu',padding='same'),
layers.BatchNormalization(),
layers.MaxPooling2D(),

layers.Flatten(),

layers.Dense(256,activation='relu'),
layers.Dropout(0.5),

layers.Dense(10,activation='softmax')

])

In [ ]:
cnn_model.compile(
optimizer='adam',
loss='sparse_categorical_crossentropy',
metrics=['accuracy']
)

In [ ]:
cnn_history=cnn_model.fit(
x_train,
y_train,
epochs=20,
batch_size=64,
validation_split=0.1,
callbacks=[early_stop]
)

# Evaluate

In [ ]:
cnn_loss,cnn_acc=cnn_model.evaluate(x_test,y_test)

# Data Augmentation


In [ ]:
data_augmentation=tf.keras.Sequential([
layers.RandomFlip("horizontal"),
layers.RandomRotation(0.1),
layers.RandomZoom(0.1)
])

# CNN with augmentation

In [ ]:
aug_model = models.Sequential([

    data_augmentation,

    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Flatten(),

    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),

    layers.Dense(10, activation='softmax')
])

In [ ]:
aug_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
aug_history = aug_model.fit(
    x_train,
    y_train,
    epochs=20,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop]
)

In [ ]:
aug_loss,aug_acc=aug_model.evaluate(x_test,y_test)

# 📈 Accuracy Learning Curves (All Models)

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(ann_history.history['accuracy'],     label='ANN Train')
plt.plot(ann_history.history['val_accuracy'], label='ANN Validation')

plt.plot(cnn_history.history['accuracy'],     label='CNN Train')
plt.plot(cnn_history.history['val_accuracy'], label='CNN Validation')

plt.plot(aug_history.history['accuracy'],     label='CNN+Aug Train')
plt.plot(aug_history.history['val_accuracy'], label='CNN+Aug Validation')

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("ANN vs CNN vs CNN+Augmentation — Accuracy")
plt.legend()
plt.grid(True)
plt.show()

# 📉 Loss Learning Curves (All Models)

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(ann_history.history['loss'],     label='ANN Train Loss')
plt.plot(ann_history.history['val_loss'], label='ANN Validation Loss')

plt.plot(cnn_history.history['loss'],     label='CNN Train Loss')
plt.plot(cnn_history.history['val_loss'], label='CNN Validation Loss')

plt.plot(aug_history.history['loss'],     label='CNN+Aug Train Loss')
plt.plot(aug_history.history['val_loss'], label='CNN+Aug Validation Loss')

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("ANN vs CNN vs CNN+Augmentation — Loss")
plt.legend()
plt.grid(True)
plt.show()

# Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

pred = cnn_model.predict(x_test)
pred = np.argmax(pred, axis=1)

# FIX: flatten y_test from shape (10000,1) to (10000,)
y_test_flat = y_test.flatten()

cm = confusion_matrix(y_test_flat, pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm,
            annot=True,
            fmt='d',
            xticklabels=class_names,
            yticklabels=class_names,
            cmap='Blues')

plt.title("CNN Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

# Classification Report

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test_flat,
    pred,
    target_names=class_names
))

# 📊 Final Comparison Table

In [ ]:
comparison=pd.DataFrame({

"Model":[
"ANN",
"CNN",
"CNN + Data Augmentation"
],

"Accuracy":[
ann_acc,
cnn_acc,
aug_acc
],

"Loss":[
ann_loss,
cnn_loss,
aug_loss
]

})

comparison

# 🔍 Analysis

## ANN (Artificial Neural Network)
- Images are flattened into vectors — **spatial information is lost**.
- Training is faster due to simpler architecture.
- Lower accuracy compared to CNN.
- More prone to overfitting on image data.

## CNN (Convolutional Neural Network)
- Learns edges, corners, and textures via convolutional filters.
- **Preserves spatial relationships** between pixels.
- Higher classification accuracy.
- Better feature extraction and generalization.

## CNN + Data Augmentation
- Random transformations (flip, rotate, zoom) create diverse training samples.
- **Reduces overfitting** by exposing the model to more variation.
- Improves robustness to unseen data.
- Slightly longer training time.
- Achieves the **highest validation accuracy** among all three models.


# Final Conclusion
This project compared ANN and CNN architectures on the CIFAR-10 dataset. ANN achieved reasonable performance but struggled because flattening images removes spatial information. CNN significantly outperformed ANN by learning hierarchical image features through convolutional layers. Applying Batch Normalization, Dropout, Early Stopping, and Data Augmentation further improved the CNN's generalization and reduced overfitting. The experimental results demonstrate that CNNs are the preferred architecture for image classification tasks and form the foundation of modern computer vision systems.